In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

### 1. Installation and Import Libraries

In [2]:
!pip install yfinance -q

In [3]:
import numpy as np
import pandas as pd
import yfinance as yf

### 2. Define the study period

In [4]:
start_date = "2020-01-01"
end_date = "2026-01-01"

### 3. Download S&P 500 and VIX

In [5]:
#^GSPC is S&P 500 Index and provies information about the overall direction of the US equity market.
#^VIX is CBOE volatility index (a real-time market indeex that measures expected **30-D stock market volatity**, known as "fear index")
market_raw = yf.download(
    tickers=["^GSPC", "^VIX"],
    start=start_date,
    end=end_date,
    auto_adjust=False,
    progress=False
)


1 Failed download:
['^VIX']: OperationalError('database is locked')


In [6]:
print ("Download shape:", market_raw.shape)
display(market_raw.head())
print(market_raw.columns)

Download shape: (1508, 12)


Price         Adj Close             Close              High               Low  \
Ticker            ^GSPC ^VIX        ^GSPC ^VIX        ^GSPC ^VIX        ^GSPC   
Date                                                                            
2020-01-02  3257.850098  NaN  3257.850098  NaN  3258.139893  NaN  3235.530029   
2020-01-03  3234.850098  NaN  3234.850098  NaN  3246.149902  NaN  3222.340088   
2020-01-06  3246.280029  NaN  3246.280029  NaN  3246.840088  NaN  3214.639893   
2020-01-07  3237.179932  NaN  3237.179932  NaN  3244.909912  NaN  3232.429932   
2020-01-08  3253.050049  NaN  3253.050049  NaN  3267.070068  NaN  3236.669922   

Price                   Open           Volume       
Ticker     ^VIX        ^GSPC ^VIX       ^GSPC ^VIX  
Date                                                
2020-01-02  NaN  3244.669922  NaN  3459930000  NaN  
2020-01-03  NaN  3226.360107  NaN  3484700000  NaN  
2020-01-06  NaN  3217.550049  NaN  3702460000  NaN  
2020-01-07  NaN  3241.860107  NaN  3435910000  NaN  
2020-01-08  NaN  3238.590088  NaN  3726840000  NaN

MultiIndex([('Adj Close', '^GSPC'),
            ('Adj Close',  '^VIX'),
            (    'Close', '^GSPC'),
            (    'Close',  '^VIX'),
            (     'High', '^GSPC'),
            (     'High',  '^VIX'),
            (      'Low', '^GSPC'),
            (      'Low',  '^VIX'),
            (     'Open', '^GSPC'),
            (     'Open',  '^VIX'),
            (   'Volume', '^GSPC'),
            (   'Volume',  '^VIX')],
           names=['Price', 'Ticker'])


### 4. Inspect MultiIndex columns

In [7]:
market_context = market_raw[[
    ("Close", "^GSPC"),
    ("Close", "^VIX")
]].copy()

market_context.columns= [
    "market_close",
    "vix_close"
]

market_context = (
    market_context
    .reset_index()
    .rename(columns={"Date": "date"})
)

display(market_context.head())

,date,market_close,vix_close
0,2020-01-02,3257.850098,NaN
1,2020-01-03,3234.850098,NaN
2,2020-01-06,3246.280029,NaN
3,2020-01-07,3237.179932,NaN
4,2020-01-08,3253.050049,NaN


In [8]:
print("Shape:", market_context.shape)

print("\nColumns:")
print(market_context.columns.tolist())

print("\nDate range:")
print(market_context["date"].min(), "to", market_context["date"].max())

print("\nMissing values:")
print(market_context.isna().sum())

Shape: (1508, 3)

Columns:
['date', 'market_close', 'vix_close']

Date range:
2020-01-02 00:00:00 to 2025-12-31 00:00:00

Missing values:
date               0
market_close       0
vix_close       1508
dtype: int64


### 5. Create market Return and VIX change

In [9]:
# Calculate daily S&P 500 market return.
market_context["market_return"] = (
    market_context["market_close"]
    .pct_change()
)

# Calculate the daily change in VIX.
market_context["vix_change"] = (
    market_context["vix_close"]
    .pct_change()
)

display(market_context.head())

/tmp/ipykernel_16/2943270729.py:10: FutureWarning: The default fill_method='pad' in Series.pct_change is deprecated and will be removed in a future version. Either fill in any non-leading NA values prior to calling pct_change or specify 'fill_method=None' to not fill NA values.
  .pct_change()


,date,market_close,vix_close,market_return,vix_change
0,2020-01-02,3257.850098,NaN,NaN,NaN
1,2020-01-03,3234.850098,NaN,-0.007060,NaN
2,2020-01-06,3246.280029,NaN,0.003533,NaN
3,2020-01-07,3237.179932,NaN,-0.002803,NaN
4,2020-01-08,3253.050049,NaN,0.004902,NaN


In [10]:
# Shift market-context variables by one trading day so that the model
# uses information available before the prediction date.

market_context["market_return_lag_1"] = (
    market_context["market_return"]
    .shift(1)
)

market_context["vix_close_lag_1"] = (
    market_context["vix_close"]
    .shift(1)
)

market_context["vix_change_lag_1"] = (
    market_context["vix_change"]
    .shift(1)
)

display(market_context.head())

,date,market_close,vix_close,market_return,vix_change,market_return_lag_1,vix_close_lag_1,vix_change_lag_1
0,2020-01-02,3257.850098,NaN,NaN,NaN,NaN,NaN,NaN
1,2020-01-03,3234.850098,NaN,-0.007060,NaN,NaN,NaN,NaN
2,2020-01-06,3246.280029,NaN,0.003533,NaN,-0.007060,NaN,NaN
3,2020-01-07,3237.179932,NaN,-0.002803,NaN,0.003533,NaN,NaN
4,2020-01-08,3253.050049,NaN,0.004902,NaN,-0.002803,NaN,NaN


In [11]:
print("Final columns:")
print(market_context.columns.tolist())

print("\nShape:")
print(market_context.shape)

print("\nDate range:")
print(market_context["date"].min(), "to", market_context["date"].max())

print("\nDuplicate dates:")
print(market_context.duplicated(subset=["date"]).sum())

print("\nMissing values:")
print(market_context.isna().sum())

Final columns:
['date', 'market_close', 'vix_close', 'market_return', 'vix_change', 'market_return_lag_1', 'vix_close_lag_1', 'vix_change_lag_1']

Shape:
(1508, 8)

Date range:
2020-01-02 00:00:00 to 2025-12-31 00:00:00

Duplicate dates:
0

Missing values:
date                      0
market_close              0
vix_close              1508
market_return             1
vix_change             1508
market_return_lag_1       2
vix_close_lag_1        1508
vix_change_lag_1       1508
dtype: int64


In [12]:
print("Market return summary:")
display(
    market_context["market_return"]
    .describe()
)

print("VIX close summary:")
display(
    market_context["vix_close"]
    .describe()
)

print("VIX change summary:")
display(
    market_context["vix_change"]
    .describe()
)

Market return summary:


count    1507.000000
mean        0.000580
std         0.013183
min        -0.119841
25%        -0.004835
50%         0.000969
75%         0.006871
max         0.095154
Name: market_return, dtype: float64

VIX close summary:


count    0.0
mean     NaN
std      NaN
min      NaN
25%      NaN
50%      NaN
75%      NaN
max      NaN
Name: vix_close, dtype: float64

VIX change summary:


count    0.0
mean     NaN
std      NaN
min      NaN
25%      NaN
50%      NaN
75%      NaN
max      NaN
Name: vix_change, dtype: float64

In [13]:
market_context_processed = market_context[
    [
        "date",
        "market_close",
        "market_return",
        "vix_close",
        "vix_change",
        "market_return_lag_1",
        "vix_close_lag_1",
        "vix_change_lag_1"
    ]
].copy()

display(market_context_processed.head())

,date,market_close,market_return,vix_close,vix_change,market_return_lag_1,vix_close_lag_1,vix_change_lag_1
0,2020-01-02,3257.850098,NaN,NaN,NaN,NaN,NaN,NaN
1,2020-01-03,3234.850098,-0.007060,NaN,NaN,NaN,NaN,NaN
2,2020-01-06,3246.280029,0.003533,NaN,NaN,-0.007060,NaN,NaN
3,2020-01-07,3237.179932,-0.002803,NaN,NaN,0.003533,NaN,NaN
4,2020-01-08,3253.050049,0.004902,NaN,NaN,-0.002803,NaN,NaN


In [14]:
output_file = (
    "/kaggle/working/"
    "market_context_processed_2020_2025.csv"
)

market_context_processed.to_csv(
    output_file,
    index=False
)

print("Saved:", output_file)
print("Final shape:", market_context_processed.shape)

Saved: /kaggle/working/market_context_processed_2020_2025.csv
Final shape: (1508, 8)


In [15]:
market = pd.read_csv("/kaggle/working/market_context_processed_2020_2025.csv")
market.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1508 entries, 0 to 1507
Data columns (total 8 columns):
 #   Column               Non-Null Count  Dtype  
---  ------               --------------  -----  
 0   date                 1508 non-null   object 
 1   market_close         1508 non-null   float64
 2   market_return        1507 non-null   float64
 3   vix_close            0 non-null      float64
 4   vix_change           0 non-null      float64
 5   market_return_lag_1  1506 non-null   float64
 6   vix_close_lag_1      0 non-null      float64
 7   vix_change_lag_1     0 non-null      float64
dtypes: float64(7), object(1)
memory usage: 94.4+ KB
